In [1]:
!pip install --upgrade pip
!pip install numpy pandas matplotlib scikit-learn nltk


zsh:1: /Users/krishnasharma/Desktop/sms_spam-nonspam_classifier/venv/bin/pip: bad interpreter: /Users/krishnasharma/sms-spam-classifier/sms_spam-nonspam_classifier/venv/bin/python: no such file or directory
  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.1
    Uninstalling pip-25.1:
      Successfully uninstalled pip-25.1
zsh:1: /Users/krishnasharma/Desktop/sms_spam-nonspam_classifier/venv/bin/pip: bad interpreter: /Users/krishnasharma/sms-spam-classifier/sms_spam-nonspam_classifier/venv/bin/python: no such file or directory
  Using cached pandas-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-no

In [2]:
import pandas as pd
import numpy as np
import re
import string

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/krishnasharma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
# 1.Load the dataset
# 2.Check missing values
# 3.Remove duplicates
# 4.Encode labels
# 5.Clean text
# 6.Remove stopwords
# 7.Perform stemming
# 8.Convert text into numerical vectors using TF-IDF
# 9.Split data into train/test sets
# 10.Prepare data for machine learning models

# loading dataset

In [4]:
file_path = 'spam.csv'
df = pd.read_csv(file_path, encoding='latin-1')

In [5]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.info()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  5572 non-null   str  
 1   Message   5572 non-null   str  
dtypes: str(2)
memory usage: 543.5 KB


(5572, 2)

In [7]:
df.isnull().sum()

Category    0
Message     0
dtype: int64

# Removing duplicate

In [8]:
df.duplicated().sum()

np.int64(415)

In [9]:
df = df.drop_duplicates(keep='first')
df.shape

(5157, 2)

# Encode label

In [10]:
encoder = LabelEncoder()
df['Category'] = encoder.fit_transform(df['Category'])
df.head()

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


# Text processing

In [11]:
ps = PorterStemmer()


def transform_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenization
    words = text.split()

    # Remove stopwords and stemming
    processed_words = []

    for word in words:
        if word not in stopwords.words('english'):
            stemmed_word = ps.stem(word)
            processed_words.append(stemmed_word)

    return ' '.join(processed_words)

# Applying text cleaning

In [12]:
# Creating new processed column

df['processed_message'] = df['Message'].apply(transform_text)

df.head()

,Category,Message,processed_message
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf live around though


# Feature Extraction

In [13]:
vectorizer = TfidfVectorizer(max_features=3000)

X = vectorizer.fit_transform(df['processed_message']).toarray()

# Target column

y = df['Category']

print('Feature matrix shape:', X.shape)
print('Target shape:', y.shape)

Feature matrix shape: (5157, 3000)
Target shape: (5157,)


# Train Test Split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (4125, 3000)
X_test shape: (1032, 3000)


# Model building

In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## For MultinomialNB

In [16]:
model1 = MultinomialNB()
model1.fit(X_train, y_train)

# Predictions
y_pred1 = model1.predict(X_test)

# Accuracy
print("Accuracy :", accuracy_score(y_test, y_pred1))

# Precision
print("Precision:", precision_score(y_test, y_pred1))

# Recall
print("Recall   :", recall_score(y_test, y_pred1))

# F1 Score
print("F1 Score :", f1_score(y_test, y_pred1))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred1))

# Full report
print("\nClassification Report:")
print(classification_report(y_test, y_pred1))

Accuracy : 0.9709302325581395
Precision: 1.0
Recall   : 0.765625
F1 Score : 0.8672566371681416

Confusion Matrix:
[[904   0]
 [ 30  98]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       904
           1       1.00      0.77      0.87       128

    accuracy                           0.97      1032
   macro avg       0.98      0.88      0.93      1032
weighted avg       0.97      0.97      0.97      1032



## For Logistic regression

In [17]:
model2 = LogisticRegression()
model2.fit(X_train, y_train)

# Predictions
y_pred2 = model2.predict(X_test)

# Accuracy
print("Accuracy :", accuracy_score(y_test, y_pred2))

# Precision
print("Precision:", precision_score(y_test, y_pred2))

# Recall
print("Recall   :", recall_score(y_test, y_pred2))

# F1 Score
print("F1 Score :", f1_score(y_test, y_pred2))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred2))

# Full report
print("\nClassification Report:")
print(classification_report(y_test, y_pred2))

Accuracy : 0.9583333333333334
Precision: 0.967032967032967
Recall   : 0.6875
F1 Score : 0.8036529680365296

Confusion Matrix:
[[901   3]
 [ 40  88]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       904
           1       0.97      0.69      0.80       128

    accuracy                           0.96      1032
   macro avg       0.96      0.84      0.89      1032
weighted avg       0.96      0.96      0.96      1032



## For SVC

In [18]:
model3 = SVC()
model3.fit(X_train, y_train)

# Predictions
y_pred3 = model3.predict(X_test)

# Accuracy
print("Accuracy :", accuracy_score(y_test, y_pred3))

# Precision
print("Precision:", precision_score(y_test, y_pred3))

# Recall
print("Recall   :", recall_score(y_test, y_pred3))

# F1 Score
print("F1 Score :", f1_score(y_test, y_pred3))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred3))

# Full report
print("\nClassification Report:")
print(classification_report(y_test, y_pred3))

Accuracy : 0.9748062015503876
Precision: 1.0
Recall   : 0.796875
F1 Score : 0.8869565217391304

Confusion Matrix:
[[904   0]
 [ 26 102]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       904
           1       1.00      0.80      0.89       128

    accuracy                           0.97      1032
   macro avg       0.99      0.90      0.94      1032
weighted avg       0.98      0.97      0.97      1032



## Combined comparison 

In [19]:
print("For mutlinomial Naive Bayes")
print("Accuracy :", accuracy_score(y_test, y_pred1))
print("Precision:", precision_score(y_test, y_pred1))
print(confusion_matrix(y_test, y_pred1))

print("\nFor Logistic Regression")
print("Accuracy :", accuracy_score(y_test, y_pred2))
print("Precision:", precision_score(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))

print("\nFor Support Vector Machine")
print("Accuracy :", accuracy_score(y_test, y_pred3))
print("Precision:", precision_score(y_test, y_pred3))
print(confusion_matrix(y_test, y_pred3))

For mutlinomial Naive Bayes
Accuracy : 0.9709302325581395
Precision: 1.0
[[904   0]
 [ 30  98]]

For Logistic Regression
Accuracy : 0.9583333333333334
Precision: 0.967032967032967
[[901   3]
 [ 40  88]]

For Support Vector Machine
Accuracy : 0.9748062015503876
Precision: 1.0
[[904   0]
 [ 26 102]]


In [20]:
import pickle
pickle.dump(vectorizer,open('vectorizer.pkl','wb'))
pickle.dump(model1,open('model1.pkl','wb'))
pickle.dump(model3,open('model3.pkl','wb'))